# Orbisent — Experiment 01: Training a Tiny Language Model From Scratch

*Beyond the Known.*

This notebook trains a **tiny character-level GPT-style Transformer** starting from **randomly initialized weights**.
No pretrained models, no downloaded weights and no external AI APIs are used. The model learns everything it knows
from one small public-domain text: the *Tiny Shakespeare* collection of Shakespeare's plays.

**What happens in this notebook**

1. Check whether a GPU is available.
2. Download a small public-domain text file (about 1 MB).
3. Turn characters into numbers with a very simple tokenizer.
4. Split the text into training and validation parts.
5. Build a small Transformer language model.
6. Train it to predict the next character, using cross-entropy loss.
7. Compare the loss before and after training.
8. Generate new text from a short prompt.
9. Save a checkpoint (weights + configuration) and download it.

**About free compute:** Google Colab's free tier does **not guarantee** a GPU, and sessions can be disconnected
or time-limited without warning. This notebook detects the hardware automatically. Without a GPU it falls back
to a short **CPU smoke test**, which only checks that everything runs — don't expect good text from it.
For the full run, choose *Runtime → Change runtime type → T4 GPU* (if one is offered).

## 1. Setup: import libraries and detect the hardware
Colab already ships with PyTorch, so normally nothing needs installing. The cell below installs it only if it is missing.

In [ ]:
# Colab normally has PyTorch pre-installed. If it is missing, install it (this only runs if the import fails).
try:
    import torch
except ImportError:
    !pip install -q torch
    import torch

import math
import os
import time
import urllib.request

import torch.nn as nn
from torch.nn import functional as F

# A "seed" makes random numbers repeatable, so two runs of this notebook start from the same random weights.
torch.manual_seed(1337)

# Use the GPU ("cuda") if Colab gave us one, otherwise use the CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch version:", torch.__version__)
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. The notebook will use the small CPU smoke-test configuration.")

## 2. Configuration
There are two settings:

* **`gpu`** — a small model sized for a free Colab GPU (roughly 5–15 minutes on a T4, depending on the GPU you get).
* **`smoke_test`** — a tiny model with very few steps. It finishes in about a minute on a CPU and only proves the code works.

By default the notebook picks `gpu` when a GPU is present and `smoke_test` otherwise.
Set `FORCE_SMOKE_TEST = True` to always run the quick version.

In [ ]:
FORCE_SMOKE_TEST = False  # Set to True for a quick check, even on a GPU.

CONFIGS = {
    # Small model for a free Colab GPU.
    "gpu": dict(
        block_size=128,     # How many previous characters the model can look at.
        batch_size=64,      # How many text snippets are processed at once.
        n_embd=192,         # Size of the vector that represents each character.
        n_head=6,           # Number of attention "heads" (must divide n_embd evenly).
        n_layer=4,          # Number of Transformer blocks stacked on top of each other.
        dropout=0.1,        # Randomly switches off 10% of signals during training to reduce overfitting.
        learning_rate=1e-3, # How big each learning step is.
        max_iters=3000,     # Total number of training steps.
        eval_interval=300,  # Measure the loss every this many steps.
        eval_iters=50,      # How many batches to average when measuring the loss.
    ),
    # Tiny model that runs quickly on a CPU, just to check that everything works.
    "smoke_test": dict(
        block_size=32,
        batch_size=16,
        n_embd=64,
        n_head=4,
        n_layer=2,
        dropout=0.0,
        learning_rate=1e-3,
        max_iters=200,
        eval_interval=50,
        eval_iters=10,
    ),
}

config_name = "smoke_test" if (FORCE_SMOKE_TEST or device == "cpu") else "gpu"
config = dict(CONFIGS[config_name])
print("Using configuration:", config_name)
for key, value in config.items():
    print(f"  {key} = {value}")

## 3. Load the dataset
We download *Tiny Shakespeare* (Shakespeare's works are in the public domain) from a public GitHub repository.
The file is saved in the Colab session only, not committed to GitHub.
If the download fails, a short built-in public-domain passage (Shakespeare's Sonnet 18) is used so the notebook still runs —
it is far too small for real learning, though.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "data/tinyshakespeare.txt"

# Backup text in case the download is not possible: Shakespeare's Sonnet 18 (public domain).
FALLBACK_TEXT = '''Shall I compare thee to a summer's day?
Thou art more lovely and more temperate:
Rough winds do shake the darling buds of May,
And summer's lease hath all too short a date;
Sometime too hot the eye of heaven shines,
And often is his gold complexion dimm'd;
And every fair from fair sometime declines,
By chance or nature's changing course untrimm'd;
But thy eternal summer shall not fade,
Nor lose possession of that fair thou ow'st;
Nor shall death brag thou wander'st in his shade,
When in eternal lines to time thou grow'st:
So long as men can breathe or eyes can see,
So long lives this, and this gives life to thee.
'''

os.makedirs("data", exist_ok=True)
if not os.path.exists(DATA_PATH):
    try:
        print("Downloading dataset...")
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    except Exception as error:
        print("Download failed:", error)
        print("Using the short built-in fallback text instead.")

if os.path.exists(DATA_PATH):
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        text = f.read()
else:
    text = FALLBACK_TEXT * 50  # Repeat it so there is enough text to form batches.

print(f"Dataset length: {len(text):,} characters")
print("First 300 characters:\n")
print(text[:300])

## 4. A simple character tokenizer
Neural networks work with numbers, not letters. Our tokenizer gives every distinct character in the text its own number
(for example `'a' → 39`). *Encoding* turns text into numbers; *decoding* turns numbers back into text.

In [ ]:
chars = sorted(set(text))  # Every distinct character, in a fixed order.
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}  # "string to integer"
itos = {i: ch for i, ch in enumerate(chars)}  # "integer to string"


def encode(s):
    # Text -> list of numbers. Characters never seen in the training text are skipped.
    return [stoi[c] for c in s if c in stoi]


def decode(ids):
    # List of numbers -> text.
    return "".join(itos[i] for i in ids)


print("Vocabulary size:", vocab_size)
print("Characters:", repr("".join(chars)))
print("encode('hello') ->", encode("hello"))
print("decode(encode('hello')) ->", decode(encode("hello")))

## 5. Train/validation split
We train on the first 90% of the text and keep the last 10% aside as *validation* data.
The model never learns from the validation part, so its loss there shows how well the model handles text it hasn't seen.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]
print(f"Training tokens: {len(train_data):,}  Validation tokens: {len(val_data):,}")


def get_batch(which):
    # Pick random snippets of text. x is the input; y is the same snippet shifted one character to the right,
    # so y[t] is the "next character" the model should predict after seeing x[0..t].
    source = train_data if which == "train" else val_data
    block_size, batch_size = config["block_size"], config["batch_size"]
    starts = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in starts])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)


xb, yb = get_batch("train")
print("Example input :", repr(decode(xb[0][:40].tolist())))
print("Example target:", repr(decode(yb[0][:40].tolist())))

## 6. The Transformer model
This is a miniature version of the GPT architecture, written out by hand:

* **Embeddings** turn each character number (and its position) into a vector of numbers the model can learn.
* **Causal self-attention** lets each position look back at earlier characters (never ahead) and decide which ones matter.
* **Feed-forward layers** process each position's information further.
* **Residual connections** and **layer normalization** keep training stable.
* A final **linear layer** produces a score for every possible next character.

All weights start out random — the model knows nothing until it is trained.

In [ ]:
class CausalSelfAttention(nn.Module):
    # Several attention "heads" run side by side; each can focus on a different pattern in the text.
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)  # Makes queries, keys and values in one step.
        self.proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        # Lower-triangular mask: position t may only look at positions 0..t (no peeking at the future).
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape  # batch size, sequence length, embedding size
        q, k, v = self.qkv(x).split(C, dim=2)
        # Reshape so each head gets its own slice: (B, n_head, T, head_size)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        # Attention scores: how much each position should "pay attention" to each earlier position.
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v  # Weighted mix of the values.
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # Put the heads back together.
        return self.resid_dropout(self.proj(y))


class FeedForward(nn.Module):
    # A small two-layer neural network applied to every position separately.
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    # One Transformer block: attention, then feed-forward, each with a residual ("shortcut") connection.
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer, dropout, **unused):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)     # Which character is it?
        self.position_embedding = nn.Embedding(block_size, n_embd)  # Where in the snippet is it?
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)  # Scores ("logits") for every possible next character.
        self.apply(self._init_weights)  # Random initialization of all weights.

    def _init_weights(self, module):
        # Small random numbers from a normal distribution: the standard way GPT-style models start.
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        x = self.drop(self.token_embedding(idx) + self.position_embedding(positions))
        x = self.blocks(x)
        logits = self.lm_head(self.ln_f(x))  # Shape: (B, T, vocab_size)
        loss = None
        if targets is not None:
            # Cross-entropy loss: low when the model gives high probability to the correct next character.
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        # Repeatedly predict one more character and append it to the text.
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size :]  # The model can only look at the last block_size characters.
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # Scores for the next character only.
            probs = F.softmax(logits, dim=-1)         # Turn scores into probabilities.
            next_id = torch.multinomial(probs, num_samples=1)  # Pick one character at random, weighted by probability.
            idx = torch.cat([idx, next_id], dim=1)
        return idx


model = TinyGPT(vocab_size=vocab_size, **config).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

## 7. Loss before training
A model that guesses completely at random has a loss of about `ln(vocab_size)` (around 4.17 for 65 characters).
Our freshly initialized model should be close to that.

In [ ]:
@torch.no_grad()
def estimate_loss():
    # Average the loss over several batches for a steadier number. eval() switches dropout off while measuring.
    model.eval()
    results = {}
    for split_name in ("train", "val"):
        losses = torch.zeros(config["eval_iters"])
        for i in range(config["eval_iters"]):
            x, y = get_batch(split_name)
            _, loss = model(x, y)
            losses[i] = loss.item()
        results[split_name] = losses.mean().item()
    model.train()
    return results


initial_losses = estimate_loss()
print(f"Random-guess loss ln({vocab_size}) = {math.log(vocab_size):.4f}")
print(f"Initial train loss: {initial_losses['train']:.4f}  Initial val loss: {initial_losses['val']:.4f}")

## 8. Training loop
Each step: take a batch, let the model predict the next characters, measure the cross-entropy loss,
compute how each weight should change (*backpropagation*), and nudge the weights with the AdamW optimizer.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])
history = []  # (step, train_loss, val_loss) for plotting later.
start_time = time.time()

for step in range(config["max_iters"] + 1):
    # Every eval_interval steps (and at the very end), measure and print the loss.
    if step % config["eval_interval"] == 0 or step == config["max_iters"]:
        losses = estimate_loss()
        history.append((step, losses["train"], losses["val"]))
        elapsed = time.time() - start_time
        print(f"step {step:5d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f} | {elapsed:6.1f}s")
    if step == config["max_iters"]:
        break

    x, y = get_batch("train")
    _, loss = model(x, y)                  # 1. Forward pass: predictions and loss.
    optimizer.zero_grad(set_to_none=True)  # 2. Clear old gradients.
    loss.backward()                        # 3. Backpropagation: compute new gradients.
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Keep updates from getting too large.
    optimizer.step()                       # 4. Update the weights.

print(f"Training finished in {time.time() - start_time:.1f} seconds.")

## 9. Initial vs. final loss
If training worked, both losses should be clearly lower than at the start.
A validation loss much higher than the training loss would mean the model is memorizing (overfitting).

In [ ]:
final_losses = estimate_loss()
print(f"{'':10s}{'initial':>10s}{'final':>10s}")
print(f"{'train':10s}{initial_losses['train']:10.4f}{final_losses['train']:10.4f}")
print(f"{'val':10s}{initial_losses['val']:10.4f}{final_losses['val']:10.4f}")

# Optional loss curve (matplotlib comes with Colab).
try:
    import matplotlib.pyplot as plt

    steps, train_curve, val_curve = zip(*history)
    plt.plot(steps, train_curve, label="train")
    plt.plot(steps, val_curve, label="validation")
    plt.xlabel("training step")
    plt.ylabel("cross-entropy loss")
    plt.title(f"Orbisent experiment 01 ({config_name})")
    plt.legend()
    plt.show()
except ImportError:
    print("matplotlib not available; skipping the plot.")

## 10. Generate text from a prompt
Give the model a few starting characters and let it continue. Lower `temperature` (e.g. 0.5) makes the output
safer and more repetitive; higher (e.g. 1.2) makes it more adventurous. After the smoke test, expect mostly gibberish.

In [ ]:
prompt = "ROMEO:"
temperature = 0.8
max_new_tokens = 500

model.eval()
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=max_new_tokens, temperature=temperature)
print(decode(generated[0].tolist()))

## 11. Save a checkpoint
A checkpoint stores everything needed to rebuild the model later: the learned weights, the configuration and the vocabulary.
It is saved inside the Colab session (in `checkpoints/`). **It is not committed to GitHub.**

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
CHECKPOINT_PATH = f"checkpoints/orbisent_exp01_{config_name}.pt"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": config,
    "config_name": config_name,
    "vocab": chars,  # The list of characters; rebuild stoi/itos from it.
    "initial_losses": initial_losses,
    "final_losses": final_losses,
    "trained_steps": config["max_iters"],
}
torch.save(checkpoint, CHECKPOINT_PATH)
print(f"Saved checkpoint to {CHECKPOINT_PATH} ({os.path.getsize(CHECKPOINT_PATH) / 1e6:.1f} MB)")

# Quick check: load the checkpoint into a brand-new model and make sure it gives the same predictions.
loaded = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
restored = TinyGPT(vocab_size=len(loaded["vocab"]), **loaded["config"]).to(device)
restored.load_state_dict(loaded["model_state_dict"])
restored.eval()
with torch.no_grad():
    x, _ = get_batch("val")
    same = torch.allclose(model(x)[0], restored(x)[0], atol=1e-5)
print("Reloaded model matches the trained model:", same)

## 12. Download the checkpoint
Colab deletes files when the session ends, so download the checkpoint to your computer
or copy it to your own Google Drive if you want to keep it.
You can also use the folder icon in Colab's left sidebar: right-click the file → *Download*.

In [ ]:
try:
    from google.colab import files  # Only exists inside Google Colab.

    files.download(CHECKPOINT_PATH)
except ImportError:
    print(f"Not running in Colab. The checkpoint is at: {os.path.abspath(CHECKPOINT_PATH)}")

# Optional: save to your Google Drive instead (it will ask for permission). Uncomment the lines below to use it.
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/orbisent
# !cp {CHECKPOINT_PATH} /content/drive/MyDrive/orbisent/

## What's next
This was the first step for Orbisent: proof that we can train a language model from random weights, end to end, in the cloud.
Possible follow-up experiments: a larger dataset, a better tokenizer, longer training, and comparing model sizes.
The Orbisent agent itself is not built yet.

*Beyond the Known.*